In [1]:
import pandas as pd
import numpy as np


In [2]:
# Load the NYC Airbnb Open Data (2019)
DATA_URL = "https://raw.githubusercontent.com/HamedCoding/Airbnb/main/AB_NYC_2019.csv"

df = pd.read_csv(DATA_URL)
df.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
# Check basic information
print("Rows, Columns:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)


Rows, Columns: (48895, 16)

Missing values:
 id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

Data types:
 id                                  int64
name                                  str
host_id                             int64
host_name                             str
neighbourhood_group                   str
neighbourhood                         str
latitude                          fl

In [4]:
# Capture before-cleaning statistics
rows_before = len(df)
columns_before = len(df.columns)
missing_before = int(df.isnull().sum().sum())
duplicates_before = int(df.duplicated().sum())

print("Rows before:", rows_before)
print("Columns before:", columns_before)
print("Missing values before:", missing_before)
print("Duplicates before:", duplicates_before)


Rows before: 48895
Columns before: 16
Missing values before: 20141
Duplicates before: 0


In [5]:
# Handle missing values

# A missing listing/host name is not recoverable from this dataset.
# Keep the row and use an explicit label instead of inventing a name.
df["name"] = df["name"].fillna("Unknown")
df["host_name"] = df["host_name"].fillna("Unknown")

# If there are zero reviews, reviews_per_month is naturally represented as 0.
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# Missing last_review values occur for listings without a recorded review.
# Use a documented sentinel date so the column can remain a proper datetime.
df["last_review"] = df["last_review"].fillna("1900-01-01")

print("Missing values after filling:")
print(df.isnull().sum())


Missing values after filling:
id                                0
name                              0
host_id                           0
host_name                         0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
last_review                       0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64


In [6]:
# Remove exact duplicate rows
print("Duplicates before:", duplicates_before)

df = df.drop_duplicates().copy()

print("Duplicates after:", df.duplicated().sum())


Duplicates before: 0
Duplicates after: 0


In [7]:
# Fix data types

# IDs are identifiers, not quantities, so store them as strings.
df["id"] = df["id"].astype(str)
df["host_id"] = df["host_id"].astype(str)

# Convert review date to a proper datetime type.
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

# Categorical fields
categorical_cols = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for col in categorical_cols:
    df[col] = df[col].astype("category")

print(df.dtypes)


id                                           str
name                                         str
host_id                                      str
host_name                                    str
neighbourhood_group                     category
neighbourhood                           category
latitude                                 float64
longitude                                float64
room_type                               category
price                                      int64
minimum_nights                             int64
number_of_reviews                          int64
last_review                       datetime64[us]
reviews_per_month                        float64
calculated_host_listings_count             int64
availability_365                           int64
dtype: object


In [8]:
# Validate important values

# Airbnb availability is measured in days in a 365-day year.
invalid_availability = ((df["availability_365"] < 0) | (df["availability_365"] > 365)).sum()

# Price should not be negative.
invalid_price = (df["price"] < 0).sum()

print("Invalid availability values:", int(invalid_availability))
print("Negative price values:", int(invalid_price))


Invalid availability values: 0
Negative price values: 0


In [9]:
# Descriptive statistics for numeric columns
df.describe()


,latitude,longitude,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
count,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,48895,48895.000000,48895.000000,48895.000000
mean,40.728949,-73.952170,152.720687,7.029962,23.274466,1994-05-05 21:06:35.598732,1.090910,7.143982,112.781327
min,40.499790,-74.244420,0.000000,1.000000,0.000000,1900-01-01 00:00:00,0.000000,1.000000,0.000000
25%,40.690100,-73.983070,69.000000,1.000000,1.000000,2016-03-24 00:00:00,0.040000,1.000000,0.000000
50%,40.723070,-73.955680,106.000000,3.000000,5.000000,2019-01-03 00:00:00,0.370000,1.000000,45.000000
75%,40.763115,-73.936275,175.000000,5.000000,24.000000,2019-06-19 00:00:00,1.580000,2.000000,227.000000
max,40.913060,-73.712990,10000.000000,1250.000000,629.000000,2019-07-08 00:00:00,58.500000,327.000000,365.000000
std,0.054530,0.046157,240.154170,20.510550,44.550582,NaN,1.597283,32.952519,131.622289


In [10]:
# Final cleaning summary

rows_after = len(df)
columns_after = len(df.columns)
missing_after = int(df.isnull().sum().sum())
duplicates_after = int(df.duplicated().sum())

summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing values",
        "Duplicate rows"
    ],
    "Before": [
        rows_before,
        columns_before,
        missing_before,
        duplicates_before
    ],
    "After": [
        rows_after,
        columns_after,
        missing_after,
        duplicates_after
    ]
})

summary["Change"] = summary["After"] - summary["Before"]

print(summary.to_string(index=False))


        Metric  Before  After  Change
          Rows   48895  48895       0
       Columns      16     16       0
Missing values   20141      0  -20141
Duplicate rows       0      0       0


In [11]:
# Final validation
print("Final shape:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

assert missing_after == 0, "Missing values still remain."
assert duplicates_after == 0, "Duplicate rows still remain."
assert invalid_availability == 0, "Invalid availability values found."
assert invalid_price == 0, "Negative prices found."


Final shape: (48895, 16)

Remaining missing values:
id                                0
name                              0
host_id                           0
host_name                         0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
last_review                       0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

Data types:
id                                           str
name                                         str
host_id                                      str
host_name                                    str
neighbourhood_group                     category
neighbourhood                           category
latitude                                 float64
lon

In [12]:
# Export the cleaned dataset
df.to_csv("airbnb_nyc_2019_cleaned.csv", index=False)
print("Saved: airbnb_nyc_2019_cleaned.csv")


Saved: airbnb_nyc_2019_cleaned.csv
